#### [ MNIST 손글씨 숫자 이미지 분류 ]<hr>

- 데이터 : mnist_train.csv, mnist_test.csv 
- 내__용 : 손글씨 그림을 저장한 파일. 0 ~ 9까지 숫자 데이터
- 주__제 : 숫자 0 ~ 9 이미지를 전달해서 정확하계 분류해주는 모델
- 학습종류 : 지도학습 + 분류
- 학습방법 : KNN, SVC

- **[0] 모듈 로딩**

In [ ]:
## 데이터 분석 관련
import pandas as pd
import numpy as np

## 데이터 시각화 관련
import matplotlib.pyplot as plt
import koreanize_matplotlib

## ML 교차검증 관련
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold

## ML 전처리 관련
from sklearn.preprocessing import MinMaxScaler

## ML 모델 관련
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## ML 파이프라인 관련
from sklearn.pipeline  import Pipeline 

## ML 모델 저장
import joblib, os 

- **[1] 데이터 준비 및 확인**

In [3]:
## 데이터 파일
TRAIN_FILE  = '../Data/Images/mnist/mnist_train.csv'
TEST_FILE   = '../Data/Images/mnist/mnist_test.csv'
MODEL_FILE  = '../Models/mnist_model.pkl'

In [6]:
## 데이터 로딩 => 첫번째 줄 컬럼X, 데이터 구분 쉼표(,)
trainDF = pd.read_csv(TRAIN_FILE, header=None)
testDF  = pd.read_csv(TEST_FILE, header=None)

In [12]:
## 기본 정보 확인
trainDF.info()
print( trainDF.head(2), f'값 :{trainDF.iloc[0].max()}, {trainDF.iloc[0].min()}\n', end='\n\n')

testDF.info()
print( testDF.head(2))

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, 0 to 784
dtypes: int64(785)
memory usage: 359.3 MB
   0    1    2    3    4    5    6    ...  778  779  780  781  782  783  784
0    5    0    0    0    0    0    0  ...    0    0    0    0    0    0    0
1    0    0    0    0    0    0    0  ...    0    0    0    0    0    0    0

[2 rows x 785 columns] 값 :255, 0


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 785 entries, 0 to 784
dtypes: int64(785)
memory usage: 59.9 MB
   0    1    2    3    4    5    6    ...  778  779  780  781  782  783  784
0    7    0    0    0    0    0    0  ...    0    0    0    0    0    0    0
1    2    0    0    0    0    0    0  ...    0    0    0    0    0    0    0

[2 rows x 785 columns]


In [13]:
## [해석] --------------------------------------------------------
## - 이미지 픽셀 값으로 0 ~ 255
## - 모든 컬럼이 숫자값
## - 타겟 컬럼은 0번 컬럼
## ---------------------------------------------------------------

- **[2] 학습용|테스트용 데이터 준비**

In [15]:
## 학습용/테스트용 데이터 파일 분리 =>완료
## 피쳐와 타겟 분리
X_train = trainDF[ trainDF.columns[1:] ]
y_train = trainDF[ trainDF.columns[0 ] ]

X_test = testDF[ testDF.columns[1:] ]
y_test = testDF[ testDF.columns[0 ] ]

print(f'[Train] {X_train.shape}, {y_train.shape}')
print(f'[TEST]  {X_test.shape}, {y_test.shape}')

[Train] (60000, 784), (60000,)
[TEST]  (10000, 784), (10000,)


- **[3] 데이터 분석 및 전처리**
    - 클래스 균형/불균형 체크
    - 피쳐 전처리 방법 결정

In [21]:
## => 클래스 균형-불균형 체크
print(f'[Train Class]\n{y_train.value_counts().to_list()}')
print(f'{(y_train.value_counts(normalize=True)).round(2).to_list()}')

## => 클래스별 데이터 수 균형!!! 

[Train Class]
[6742, 6265, 6131, 5958, 5949, 5923, 5918, 5851, 5842, 5421]
[0.11, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.09]


In [28]:
## => 1개 행만 추출 피쳐 확인
X_train.iloc[0].describe()

## => 0~ 255 범위 값 ==> 0 ~ 1 사이로 스케일링 : MinMaxScaler

count    784.000000
mean      35.108418
std       79.699674
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      255.000000
Name: 0, dtype: float64

- **[4] 전처리 + 교차검증 + 튜닝**

In [ ]:
## 파이프라인 인스턴스 : 전처리 인스턴스, 모델 인스턴스  
pipe = Pipeline(steps=[
    ('mmScaler', MinMaxScaler()),
    ('kModel', KNeighborsClassifier())
])

## 모델 파라미터 Dict : 모델변수명_ _파라미터 
params = {'kModel__n_neighbors':range(3, 51,2),
          'kModel__weights':['uniform', 'distance'],
          'kModel__algorithm':['auto', 'ball_tree', 'kd_tree', 'brute']}

## 교차검증 인스턴스
skFold = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)

## 튜닝 인스턴스
gsTunning = GridSearchCV(pipe, param_grid=params, cv=skFold)

## 교차검증 + 튜닝 진행
gsTunning.fit(X_train, y_train)

In [ ]:
## 교차검증 + 튜닝 결과 분석
cvDF = gsTunning.
cvDF.columns

AttributeError: 'GridSearchCV' object has no attribute 'cv_results_'

- **[5]성능 평가**

- **[6]모델 저장**